# Hybrid 2.0: Transformer Embeddings + LightGBM (Walk-Forward + Export)

**Architecture:**
- Transformer pre-trained as AutoEncoder (Reconstruction of 50 candles)
- 64-dim embeddings + tabular indicators -> LightGBM Regressor
- Strict Walk-Forward Validation (Zero Leakage)
- Auto Export Models for Live Trading


## 0. Data Pipeline


In [ ]:
import os, sys, glob
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

TIMEFRAME = '1d'
DATA_DIR = Path('/kaggle/input/datasets/hungbui317/macd-coin/data')
OHLCV_DIR = DATA_DIR / 'ohlcv'
OUTPUT_FILE = f'/kaggle/working/features_{TIMEFRAME}_full.parquet'

TF_CONFIG = {
    '4h':  {'rule': '4h',  'min_bars': 200, 'atr_clamp': (0.002, 0.06), 'max_tp': 0.15, 'max_bars': 30, 'unit': '4h bars'},
    '8h':  {'rule': '8h',  'min_bars': 100, 'atr_clamp': (0.003, 0.08), 'max_tp': 0.20, 'max_bars': 20, 'unit': '8h bars'},
    '12h': {'rule': '12h', 'min_bars': 80,  'atr_clamp': (0.004, 0.10), 'max_tp': 0.25, 'max_bars': 15, 'unit': '12h bars'},
    '1d':  {'rule': '1D',  'min_bars': 100, 'atr_clamp': (0.005, 0.13), 'max_tp': 0.30, 'max_bars': 15, 'unit': 'days'},
}
CFG = TF_CONFIG[TIMEFRAME]
print(f"Timeframe: {TIMEFRAME} | Output: {OUTPUT_FILE}")
if OHLCV_DIR.exists(): print(f"Found {len(list(OHLCV_DIR.glob('*.parquet')))} symbol files")
else: print(f"⚠️ {OHLCV_DIR} not found!")


### Pipeline Functions


In [ ]:
def load_ohlcv_1h(symbol):
    for name in [f"{symbol}_USDT.parquet", f"{symbol}.parquet"]:
        fp = OHLCV_DIR / name
        if fp.exists():
            df = pd.read_parquet(fp)
            if 'timestamp' not in df.columns and 'open_time' in df.columns: df = df.rename(columns={'open_time':'timestamp'})
            df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms') if df['timestamp'].dtype=='int64' else pd.to_datetime(df['timestamp'])
            return df.sort_values('timestamp').reset_index(drop=True)
    return pd.DataFrame()
def resample_1h(df_1h, tf):
    if df_1h.empty: return pd.DataFrame()
    return df_1h.set_index('timestamp').resample(TF_CONFIG[tf]['rule']).agg({'open':'first','high':'max','low':'min','close':'last','volume':'sum'}).dropna().reset_index()
def calculate_rsi(prices, period=14):
    d = prices.diff(); g = d.where(d>0,0).rolling(period).mean(); l = (-d.where(d<0,0)).rolling(period).mean()
    return 100-(100/(1+g/(l.replace(0,np.nan)+1e-9)))
def calculate_macd(df, fast=12, slow=26, signal=9):
    ef=df['close'].ewm(span=fast).mean(); es=df['close'].ewm(span=slow).mean()
    df['macd']=ef-es; df['macd_signal']=df['macd'].ewm(span=signal).mean(); df['macd_histogram']=df['macd']-df['macd_signal']
    df['macd_cross_up']=((df['macd']>df['macd_signal'])&(df['macd'].shift(1)<=df['macd_signal'].shift(1))).astype(int)
    df['macd_cross_down']=((df['macd']<df['macd_signal'])&(df['macd'].shift(1)>=df['macd_signal'].shift(1))).astype(int)
    df['macd_slope']=df['macd'].diff(); df['macd_acceleration']=df['macd_slope'].diff(); return df
def calculate_liquidity_sweep(df, lookback=20):
    df = df.copy()
    df['swing_low'] = df['low'].rolling(window=lookback).min().shift(1)
    df['swing_high'] = df['high'].rolling(window=lookback).max().shift(1)
    df['candle_range'] = df['high'] - df['low'] + 1e-9
    df['lower_wick'] = df[['open', 'close']].min(axis=1) - df['low']
    df['upper_wick'] = df['high'] - df[['open', 'close']].max(axis=1)
    df['lower_wick_ratio'] = df['lower_wick'] / df['candle_range']
    df['upper_wick_ratio'] = df['upper_wick'] / df['candle_range']
    df['vol_sma_20'] = df['volume'].rolling(20).mean()
    cond_sweep_bottom = df['low'] < df['swing_low']
    cond_reject_bottom = df['close'] > df['swing_low']
    cond_pinbar_bottom = df['lower_wick_ratio'] > 0.3
    cond_vol_surge = df['volume'] > df['vol_sma_20']
    df['bullish_sweep'] = (cond_sweep_bottom & cond_reject_bottom & cond_pinbar_bottom & cond_vol_surge).astype(int)
    cond_sweep_top = df['high'] > df['swing_high']
    cond_reject_top = df['close'] < df['swing_high']
    cond_pinbar_top = df['upper_wick_ratio'] > 0.3
    df['bearish_sweep'] = (cond_sweep_top & cond_reject_top & cond_pinbar_top & cond_vol_surge).astype(int)
    df['macd_cross_up'] = df['bullish_sweep']
    df['macd_cross_down'] = df['bearish_sweep']
    return df
def calculate_features(df):
    df=df.copy(); df['log_returns']=np.log(df['close']/df['close'].shift(1))
    df['high_low_range']=(df['high']-df['low'])/df['close']; df['body_size']=abs(df['close']-df['open'])/df['close']
    df['candle_range']=df['high']-df['low']+1e-9; df['lower_wick']=df[['open','close']].min(axis=1)-df['low']; df['upper_wick']=df['high']-df[['open','close']].max(axis=1)
    df['lower_wick_ratio_current']=df['lower_wick']/df['candle_range']; df['upper_wick_ratio_current']=df['upper_wick']/df['candle_range']
    for p in [7,14,21,50,100,200]: df[f'ema_{p}']=df['close'].ewm(span=p).mean()
    for p in [10,20,50,200]: df[f'sma_{p}']=df['close'].rolling(p).mean()
    tr=pd.concat([df['high']-df['low'],abs(df['high']-df['close'].shift(1)),abs(df['low']-df['close'].shift(1))],axis=1).max(axis=1)
    df['atr_14']=tr.rolling(14).mean(); df['volatility_14']=df['log_returns'].rolling(14).std()
    df['vol_sma_14']=df['volatility_14'].rolling(14).mean(); df['vol_compression']=df['volatility_14']/(df['vol_sma_14']+1e-9)
    df['volume_sma_20']=df['volume'].rolling(20).mean(); df['volume_std_20']=df['volume'].rolling(20).std()
    df['volume_ratio']=df['volume']/(df['volume_sma_20']+1e-9); df['volume_zscore']=(df['volume']-df['volume_sma_20'])/(df['volume_std_20']+1e-9)
    df['volume_trend']=df['volume'].rolling(7).mean()/(df['volume'].rolling(21).mean()+1e-9); df['volume_spike']=(df['volume_ratio']>2).astype(int)
    df['rsi_14']=calculate_rsi(df['close'], 14); df['rsi_slope']=df['rsi_14'].diff(3)
    l14=df['low'].rolling(14).min(); h14=df['high'].rolling(14).max()
    df['stoch_k']=100*(df['close']-l14)/(h14-l14).replace(0,np.nan); df['stoch_d']=df['stoch_k'].rolling(3).mean()
    df['roc_7']=df['close'].pct_change(7); df['roc_14']=df['close'].pct_change(14)
    # Phase 11 Features
    df['sma_30']=df['close'].rolling(30).mean(); df['price_vs_sma_30']=df['close']/(df['sma_30']+1e-9)
    df['momentum_30']=df['close'].pct_change(30)
    pdm=df['high'].diff(); mdm=-df['low'].diff()
    pdm=pdm.where((pdm>mdm)&(pdm>0),0); mdm=mdm.where((mdm>pdm)&(mdm>0),0); atr_s=tr.rolling(14).mean()
    pdi=100*(pdm.rolling(14).mean()/atr_s.replace(0,np.nan)); mdi=100*(mdm.rolling(14).mean()/atr_s.replace(0,np.nan))
    df['adx']=(100*abs(pdi-mdi)/(pdi+mdi).replace(0,np.nan)).rolling(14).mean()
    df['dist_to_high_30d']=(df['close']-df['high'].rolling(30).max())/df['close']
    df['dist_to_low_30d']=(df['close']-df['low'].rolling(30).min())/df['close']
    for e in [21,50,200]: df[f'dist_to_ema_{e}_pct']=(df['close']-df[f'ema_{e}'])/df['close']
    df['trend_state']=np.where(df['close']>df['sma_50'],1,np.where(df['close']<df['sma_50'],-1,0))
    df['is_trending']=(df['adx']>25).astype(int); df['is_volatile']=(df['vol_compression']>1.5).astype(int)
    df['hour_sin']=np.sin(2*np.pi*df['timestamp'].dt.hour/24); df['hour_cos']=np.cos(2*np.pi*df['timestamp'].dt.hour/24)
    df['day_sin']=np.sin(2*np.pi*df['timestamp'].dt.dayofweek/7); df['day_cos']=np.cos(2*np.pi*df['timestamp'].dt.dayofweek/7)
    df['vol_ratio_alpha']=df['volume_ratio']*df['volatility_14']
    df=calculate_macd(df); df=df.drop(columns=['macd_cross_up','macd_cross_down'], errors='ignore')
    df=calculate_liquidity_sweep(df); return df.dropna(subset=['macd','swing_low','vol_sma_20'])
def _label_triple_barrier(df,tp_pct,sl_pct,max_bars,use_atr,atr_tp_mult,atr_sl_mult,min_tp,max_tp):
    df=df.copy(); n=len(df); close=df['close'].values; high=df['high'].values; low=df['low'].values
    is_long=(df['macd_cross_up']==1).values; atr=df['atr_14'].values if 'atr_14' in df.columns else np.zeros(n)
    ci=np.where((df['macd_cross_up']==1)|(df['macd_cross_down']==1))[0]; ci=ci[ci<n-max_bars]
    df['label']=np.nan; df['trade_result']=''
    for idx in ci:
        e=close[idx]; lo_dir=is_long[idx]
        # Structural Swing High/Low (30 bars)
        lb_start=max(0,idx-30); s_h,s_l=high[lb_start:idx+1].max(),low[lb_start:idx+1].min()
        atr_val = atr[idx] if atr[idx] > 0 else (e * 0.02)
        buf = atr_val * 0.5
        if lo_dir: 
            sl=s_l-buf; risk=e-sl; tl=max(s_h, e+risk*2.0)
        else: 
            sl=s_h+buf; risk=sl-e; tl=min(s_l, e-risk*2.0)
        at,asl=abs(tl-e)/(e+1e-9),abs(e-sl)/(e+1e-9)
        if at/(asl+1e-9)<1.5: # R:R Filter
            df.iloc[idx,df.columns.get_loc('label')]=0.5; df.iloc[idx,df.columns.get_loc('trade_result')]='NOISE_RR'; continue
        fh=high[idx+1:idx+1+max_bars]; fl=low[idx+1:idx+1+max_bars]
        if len(fh)==0: continue
        mp=((fh-e)/e).max() if lo_dir else ((e-fl.min())/e if len(fl)>0 else 0)
        md=((e-fl.min())/e if len(fl)>0 else 0) if lo_dir else ((fh-e)/e).max()
        th=np.where(fh>=tl)[0] if lo_dir else np.where(fl<=tl)[0]; sh=np.where(fl<=sl)[0] if lo_dir else np.where(fh>=sl)[0]
        if len(th)>0 and (len(sh)==0 or th[0]<=sh[0]):
            r,pnl_r='TP_HIT',1.1
        elif len(sh)>0:
            r,pnl_r='SL_HIT',-1.0
        else:
            r='TIMEOUT'; pnl_r=(mp/(at+1e-9))-(md/(asl+1e-9))
        score=3.0*pnl_r-2.5*(md/(asl+1e-9)); lbl=np.clip(1/(1+np.exp(-score)),0.05,0.95)
        df.iloc[idx,df.columns.get_loc('label')]=float(lbl); df.iloc[idx,df.columns.get_loc('trade_result')]=r
    return df
def generate_labels(df,tp_pct=0.10,sl_pct=0.05,max_bars=15,use_atr=True,atr_tp_mult=4.0,atr_sl_mult=2.0,min_tp=0.01,max_tp=0.30):
    if 'symbol' in df.columns:
        return df.groupby('symbol',group_keys=False).apply(lambda x:_label_triple_barrier(x.sort_values('timestamp'),tp_pct,sl_pct,max_bars,use_atr,atr_tp_mult,atr_sl_mult,min_tp,max_tp)).reset_index(drop=True)
    return _label_triple_barrier(df,tp_pct,sl_pct,max_bars,use_atr,atr_tp_mult,atr_sl_mult,min_tp,max_tp)
def apply_winsorization(df,fc,lo=0.01,hi=0.99):
    df=df.copy()
    for c in fc:
        if c in df.columns and df[c].dtype in ['float64','float32','int64']: l,h=df[c].quantile(lo),df[c].quantile(hi); df[c]=df[c].clip(l,h)
    return df
def apply_feature_shift(df):
    ex={'timestamp','symbol','open','high','low','close','volume','label','trade_result','macd_cross_up','macd_cross_down'}
    sc=[c for c in df.columns if c not in ex]
    if 'symbol' in df.columns: df[sc]=df.groupby('symbol')[sc].shift(1)
    else: df[sc]=df[sc].shift(1)
    return df.dropna(subset=sc[:3])
print("✓ Pipeline loaded.")


### Run Pipeline


In [ ]:
symbols=[f.stem.replace('_USDT','') for f in OHLCV_DIR.glob('*.parquet')]
symbols=[s for s in symbols if not any(x in s for x in ['-26','-25','-24'])]
print(f"Found {len(symbols)} symbols")
btc_context=pd.DataFrame()
btc_sym='BTCUSDT' if 'BTCUSDT' in symbols else ('BTC' if 'BTC' in symbols else None)
if btc_sym:
    btc_df=calculate_features(resample_1h(load_ohlcv_1h(btc_sym),TIMEFRAME))
    btc_context=btc_df[['timestamp','close','sma_200','adx','log_returns']].copy()
    btc_context.columns=['timestamp','btc_close','btc_sma_200','btc_adx','btc_returns']
    btc_context['btc_is_bull_regime']=(btc_context['btc_close']>btc_context['btc_sma_200']).astype(int)
    btc_context['btc_trend_strength']=np.where(btc_context['btc_adx']>25,1,0)
all_data=[]
for sym in symbols:
    try:
        d1=load_ohlcv_1h(sym)
        if d1.empty: continue
        dt=resample_1h(d1,TIMEFRAME)
        if len(dt)<CFG['min_bars']: continue
        dt['symbol']=sym; dt=calculate_features(dt); dt['is_bullish_cross']=dt['macd_cross_up'].values
        # Phase 11: Add Multi-Timeframe (1D) Context
        d1d=resample_1h(d1,'1d')
        d1d['ema_200_1d']=d1d['close'].ewm(span=200).mean()
        d1d['rsi_14_1d']=calculate_rsi(d1d['close'],14)
        d1d['ema_200_1d_dist']=(d1d['close']-d1d['ema_200_1d'])/d1d['close']
        d1d_feat=d1d[['timestamp','ema_200_1d_dist','rsi_14_1d']].copy()
        d1d_feat['date']=d1d_feat['timestamp'].dt.date
        d1d_feat=d1d_feat.drop(columns='timestamp').shift(1) # Prevent lookahead
        dt['date']=dt['timestamp'].dt.date
        dt=dt.merge(d1d_feat,on='date',how='left').drop(columns='date')
        for c in ['ema_200_1d_dist','rsi_14_1d']: dt[c]=dt[c].ffill().fillna(0.5 if 'rsi' in c else 0)
        
        if not btc_context.empty:
            dt=dt.merge(btc_context,on='timestamp',how='left')
            for c in ['btc_is_bull_regime','btc_trend_strength','btc_returns']: dt[c]=dt[c].ffill().fillna(0)
            dt['rs_vs_btc']=dt['log_returns']-dt['btc_returns']; dt['rs_vs_btc_sma7']=dt['rs_vs_btc'].rolling(7).mean()
            dt['btc_corr']=dt['log_returns'].rolling(14).corr(dt['btc_returns']).fillna(0)
        all_data.append(dt); print(f"  ✓ {sym}: {len(dt)} {CFG['unit']}")
    except Exception as e: print(f"  ✗ {sym}: {e}")
df=pd.concat(all_data,ignore_index=True); print(f"\n✓ Combined: {len(df)} rows")
df=apply_feature_shift(df)
fc=[c for c in df.columns if c not in ['timestamp','symbol','open','high','low','close','volume','label','trade_result']]
df=apply_winsorization(df,fc); print("Generating Soft Labels...")
df=generate_labels(df,max_bars=CFG['max_bars'],max_tp=CFG['max_tp'])
df.to_parquet(OUTPUT_FILE,index=False); print(f"\n✅ Saved to {OUTPUT_FILE} ({len(df)} rows)")
cr=df[(df['macd_cross_up']==1)|(df['macd_cross_down']==1)]; lb=cr[cr['label'].notnull()]; y=lb['label'].values
print(f"Labels: TP={int((y>=0.9).sum())}, SL={int((y<=0.1).sum())}, TO={int(((y>0.1)&(y<0.9)).sum())}, mean={y.mean():.3f}")


## 1. Config


In [ ]:
import torch, torch.nn as nn, torch.optim as optim, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler, LabelEncoder, RobustScaler
from sklearn.metrics import roc_auc_score
import lightgbm as lgb
import joblib

WINDOW_SIZE=50; BATCH_SIZE=128; PRETRAIN_EPOCHS=30; LEARNING_RATE=1e-4; WEIGHT_DECAY=0.05

SEQ_FEATURES=['log_returns','high_low_range','body_size','volatility_14','macd','macd_slope','volume_ratio','volume_zscore']
CONTEXT_FEATURES=['btc_is_bull_regime','btc_trend_strength','adx','hour_sin','hour_cos','day_sin','day_cos','btc_corr','trend_state','is_trending','is_volatile','macd_acceleration','volume_spike','vol_ratio_alpha','ema_200_1d_dist','rsi_14_1d']
SIGNAL_FEATURES=['rsi_14','rsi_slope','stoch_k','stoch_d','roc_7','roc_14','volume_ratio','volume_zscore','volume_trend','rs_vs_btc','rs_vs_btc_sma7','vol_compression','dist_to_high_30d','dist_to_low_30d','dist_to_ema_21_pct','dist_to_ema_50_pct','dist_to_ema_200_pct','price_vs_sma_30','momentum_30','macd_slope','macd_acceleration','lower_wick_ratio_current','upper_wick_ratio_current']


## 2. Transformer (Embedding Extractor)


In [ ]:
class DropPath(nn.Module):
    def __init__(self, dp=0.1):
        super().__init__(); self.dp=dp
    def forward(self, x):
        if not self.training or self.dp==0: return x
        k=1-self.dp; return x/k*torch.floor(torch.rand((x.shape[0],)+(1,)*(x.ndim-1),device=x.device)+k)

class HybridScorer(nn.Module):
    def __init__(self, seq_in_dim, context_in_dim, signal_in_dim, num_symbols=0, sym_emb_dim=16, d_model=64, nhead=4, num_layers=2, window_size=50):
        super().__init__()
        self.seq_proj = nn.Linear(seq_in_dim, d_model)
        self.pos_encoder = nn.Parameter(torch.randn(1, window_size, d_model))
        enc = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, dim_feedforward=d_model*2, batch_first=True, dropout=0.15)
        self.transformer = nn.TransformerEncoder(enc, num_layers=num_layers)
        self.ln_seq = nn.LayerNorm(d_model); self.drop_path = DropPath(0.1)
        self.has_emb = num_symbols > 0
        self.sym_emb = nn.Embedding(num_symbols, sym_emb_dim) if self.has_emb else None
        tab_dim = context_in_dim + signal_in_dim + (sym_emb_dim if self.has_emb else 0)
        self.tab_branch = nn.Sequential(nn.Linear(tab_dim, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.2), nn.Linear(128, d_model), nn.LayerNorm(d_model), nn.ReLU())
        self.cross_attn = nn.MultiheadAttention(d_model, nhead, batch_first=True, dropout=0.1)
        self.gmu = nn.Sequential(nn.Linear(d_model*2, d_model), nn.Sigmoid())
        self.classifier = nn.Sequential(nn.Linear(d_model, 32), nn.BatchNorm1d(32), nn.ReLU(), nn.Dropout(0.2), nn.Linear(32, 1))
        self.reconstruct_head = nn.Linear(d_model, seq_in_dim)

    def forward(self, seq_x, ctx_x=None, sig_x=None, sym_x=None):
        xs = self.seq_proj(seq_x) + self.pos_encoder
        xs_out = self.transformer(xs); xs_enc = self.ln_seq(self.drop_path(xs_out) + xs)
        tab = [ctx_x, sig_x] + ([self.sym_emb(sym_x)] if self.has_emb and sym_x is not None else [])
        tc = torch.cat(tab, dim=1)
        if self.training: tc = tc + torch.randn_like(tc)*0.01
        te = self.tab_branch(tc)
        sc, _ = self.cross_attn(te.unsqueeze(1), xs_enc, xs_enc); sc = sc.squeeze(1)
        z = self.gmu(torch.cat([sc, te], dim=1)); f = z*sc + (1-z)*te
        return self.classifier(f)

    def forward_ae(self, seq_x):
        """Dùng riêng cho quá trình Pre-train AutoEncoder"""
        x_seq = self.seq_proj(seq_x) + self.pos_encoder
        x_seq_out = self.transformer(x_seq)
        x_seq_encoded = self.ln_seq(self.drop_path(x_seq_out) + x_seq)
        return self.reconstruct_head(x_seq_encoded)

    def get_embeddings(self, seq_x, ctx_x, sig_x, sym_x=None):
        self.eval()
        with torch.no_grad():
            xs = self.seq_proj(seq_x) + self.pos_encoder
            return self.ln_seq(self.transformer(xs)).mean(dim=1).cpu().numpy()


## 3. Data Loading


In [ ]:
class SimpleDataset(Dataset):
    def __init__(self,s,c,g,y,l): self.s=s;self.c=c;self.g=g;self.y=y;self.l=l
    def __len__(self): return len(self.l)
    def __getitem__(self,i): return self.s[i],self.c[i],self.g[i],self.y[i],self.l[i]

def prepare_data(tf):
    paths=[f'/kaggle/working/features_{tf}_full.parquet',f'./data/processed/features_{tf}_full.parquet']
    path=next((p for p in paths if os.path.exists(p)),None)
    if not path: raise FileNotFoundError(f"No data! Run Pipeline. Looked: {paths}")
    
    # BẢN VÁ CHÍ MẠNG: Sort theo 'symbol' trước để giữ lịch sử mỗi coin liên tục
    df = pd.read_parquet(path).sort_values(['symbol', 'timestamp']).reset_index(drop=True)
    
    mask = ((df['macd_cross_up']==1)|(df['macd_cross_down']==1)) & df['label'].notnull()
    
    idx_all = []
    for i in df[mask].index:
        if i >= WINDOW_SIZE - 1:
            # BẢO VỆ RANH GIỚI: Đảm bảo 50 nến bị cắt ra thuộc CÙNG MỘT ĐỒNG COIN
            if df.loc[i, 'symbol'] == df.loc[i - WINDOW_SIZE + 1, 'symbol']:
                idx_all.append(i)
                
    print(f"Dataset prepared! Total valid valid sequences: {len(idx_all)}")
    return df, idx_all


## 4. Hybrid 2.0 Training (Walk-Forward & Final Export)


In [ ]:
def train_and_evaluate_window(df, indices_tr, indices_te, is_final_run=False):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    # 1. SCALING CHỐNG LEAKAGE (Chỉ fit trên tập Train)
    cc = [f for f in CONTEXT_FEATURES if f in df.columns]
    sc = [f for f in SIGNAL_FEATURES if f in df.columns]
    sq = [f for f in SEQ_FEATURES if f in df.columns]
    
    ctx_scaler = RobustScaler().fit(df.loc[indices_tr, cc].fillna(0))
    sig_scaler = RobustScaler().fit(df.loc[indices_tr, sc].fillna(0))
    seq_scaler = StandardScaler().fit(df.loc[indices_tr, sq].fillna(0))
    sym_encoder = LabelEncoder().fit(df['symbol']) # Mã hóa toàn bộ symbol để tránh lỗi
    
    def extract_tensors(idx):
        ctx = ctx_scaler.transform(df.loc[idx, cc].fillna(0)).astype(np.float32)
        sig = sig_scaler.transform(df.loc[idx, sc].fillna(0)).astype(np.float32)
        ss = seq_scaler.transform(df[sq].fillna(0)).astype(np.float32)
        seqs = np.array([ss[i-WINDOW_SIZE+1:i+1] for i in idx], dtype=np.float32)
        raw_y = df.loc[idx, 'label'].values.astype(np.float32)
        # Xử lý symbol lạ trong tập Test
        syms = np.array([sym_encoder.transform([s])[0] if s in sym_encoder.classes_ else 0 for s in df.loc[idx, 'symbol']])
        return seqs, ctx, sig, syms, raw_y
        
    s_tr, c_tr, g_tr, sym_tr, y_tr_raw = extract_tensors(indices_tr)
    s_te, c_te, g_te, sym_te, y_te_raw = extract_tensors(indices_te)
    
    y_tr_soft = np.clip(y_tr_raw, 0.05, 0.95)
    
    train_ds = SimpleDataset(torch.tensor(s_tr), torch.tensor(c_tr), torch.tensor(g_tr), torch.tensor(sym_tr), torch.tensor(y_tr_soft).unsqueeze(1))
    test_ds = SimpleDataset(torch.tensor(s_te), torch.tensor(c_te), torch.tensor(g_te), torch.tensor(sym_te), torch.tensor(y_te_raw).unsqueeze(1))
    
    train_loader = DataLoader(train_ds, batch_size=128, shuffle=False)
    train_shuffle = DataLoader(train_ds, batch_size=64, shuffle=True)
    test_loader = DataLoader(test_ds, batch_size=128, shuffle=False)
    
    model = HybridScorer(s_tr.shape[2], c_tr.shape[1], g_tr.shape[1], len(sym_encoder.classes_)).to(device)
    
    # === BƯỚC 1: AUTOENCODER PRE-TRAINING ===
    epochs_ae = PRETRAIN_EPOCHS if is_final_run else int(PRETRAIN_EPOCHS * 0.5)
    opt_ae = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    for epoch in range(epochs_ae):
        model.train()
        for bs,_,_,_,_ in train_shuffle:
            opt_ae.zero_grad()
            nn.MSELoss()(model.forward_ae(bs.to(device)), bs.to(device)).backward()
            opt_ae.step()
            
    # TRÍCH XUẤT EMBEDDINGS
    model.eval()
    def extract_emb(loader):
        embs = []
        with torch.no_grad():
            for bs,bc,bg,by,_ in loader:
                emb = model.get_embeddings(bs.to(device), bc.to(device), bg.to(device), by.to(device))
                embs.append(emb)
        return np.vstack(embs)
        
    emb_tr = extract_emb(train_loader)
    emb_te = extract_emb(test_loader)
    
    # === BƯỚC 2: PATH 1 (LIGHTGBM) ===
    X_tr_lgbm = np.hstack([emb_tr, c_tr, g_tr])
    X_te_lgbm = np.hstack([emb_te, c_te, g_te])
    
    lgbm_tab = lgb.LGBMClassifier(n_estimators=500, learning_rate=0.03, max_depth=4, class_weight='balanced', random_state=42, n_jobs=-1, verbose=-1)
    lgbm_tab.fit(X_tr_lgbm, (y_tr_raw >= 0.5).astype(int))
    lgbm_preds_te = lgbm_tab.predict_proba(X_te_lgbm)[:, 1]
    
    # === BƯỚC 3: PATH 2 (FINE-TUNE TRANSFORMER) ===
    opt_ft = optim.AdamW([{'params': model.seq_proj.parameters(), 'lr': 1e-5},
                          {'params': model.transformer.parameters(), 'lr': 1e-5},
                          {'params': model.tab_branch.parameters(), 'lr': 5e-5},
                          {'params': model.classifier.parameters(), 'lr': 1e-4}], weight_decay=0.05)
    
    # Tính trọng số mất cân bằng (Weighting) để Transformer mạnh dạn hơn
    num_pos = (y_tr_raw >= 0.5).sum()
    num_neg = len(y_tr_raw) - num_pos
    weight_val = num_neg / (num_pos + 1e-9)
    # Giới hạn weight_val để tránh loss bùng nổ
    weight_val = min(max(weight_val, 1.0), 10.0) 
    pos_weight = torch.tensor([weight_val], dtype=torch.float32).to(device)
    
    criterion_ft = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    
    epochs_ft = 15 if is_final_run else 8
    for epoch in range(epochs_ft):
        model.train()
        for bs,bc,bg,by,bl in train_shuffle:
            opt_ft.zero_grad()
            loss = criterion_ft(model(bs.to(device),bc.to(device),bg.to(device),by.to(device)), bl.to(device))
            loss.backward()
            opt_ft.step()
            
    model.eval()
    nn_preds_te = []
    with torch.no_grad():
        for bs,bc,bg,by,_ in test_loader:
            p = torch.sigmoid(model(bs.to(device),bc.to(device),bg.to(device),by.to(device)))
            nn_preds_te.extend(p.cpu().numpy())
    nn_preds_te = np.array(nn_preds_te).flatten()
    
    # === ĐÁNH GIÁ ĐỒNG THUẬN (ENSEMBLE) ===
    hm = (y_te_raw >= 0.9) | (y_te_raw <= 0.1)
    if hm.sum() < 5: return 0.5, 0.5, 0.5, None, None, None
    
    y_hard = (y_te_raw[hm] >= 0.9).astype(int)
    auc_l = roc_auc_score(y_hard, lgbm_preds_te[hm])
    auc_n = roc_auc_score(y_hard, nn_preds_te[hm])
    
    ens_preds = (lgbm_preds_te + nn_preds_te) / 2.0
    auc_e = roc_auc_score(y_hard, ens_preds[hm])
    
    scalers_dict = {'ctx': ctx_scaler, 'sig': sig_scaler, 'seq': seq_scaler, 'sym': sym_encoder}
    return auc_l, auc_n, auc_e, lgbm_tab, model, scalers_dict

def train_hybrid_lgbm(tf='1d'):
    print(f"\n{'='*60}\nKHỞI ĐỘNG HỆ THỐNG QUANT: WALK-FORWARD & DEPLOY ({tf})\n{'='*60}")
    
    df, idx_all = prepare_data(tf)
    timestamps = df.loc[idx_all, 'timestamp']
    start_date = timestamps.min()
    end_date = timestamps.max()
    
    # ---------------------------------------------------------
    # PHASE A: WALK-FORWARD VALIDATION
    # ---------------------------------------------------------
    print(f"\n[PHASE A] Chạy Walk-Forward Validation ({start_date.date()} -> {end_date.date()})")
    curr_date = start_date
    train_months = 6
    test_months = 2
    wf_results = []
    
    while True:
        train_end = curr_date + pd.DateOffset(months=train_months)
        test_end = train_end + pd.DateOffset(months=test_months)
        if test_end > end_date: break
            
        i_tr = timestamps[(timestamps >= curr_date) & (timestamps < train_end)].index.tolist()
        i_te = timestamps[(timestamps >= train_end) & (timestamps < test_end)].index.tolist()
        
        if len(i_tr) > 200 and len(i_te) > 20:
            print(f"📍 Cửa sổ: Train ({curr_date.date()}->{train_end.date()}) | Test ({train_end.date()}->{test_end.date()})")
            auc_l, auc_n, auc_e, _, _, _ = train_and_evaluate_window(df, i_tr, i_te, is_final_run=False)
            print(f"   >>> OOS Hard AUC | LGBM: {auc_l:.4f} | NN: {auc_n:.4f} | ENS: {auc_e:.4f}")
            wf_results.append(auc_e)
            
        curr_date += pd.DateOffset(months=test_months)
        
    if wf_results:
        print(f"\n✅ TỔNG KẾT WALK-FORWARD: Ensemble OOS AUC trung bình: {np.mean(wf_results):.4f} (Độ lệch: {np.std(wf_results):.4f})")
    
    # ---------------------------------------------------------
    # PHASE B: TRAIN FINAL MODEL VÀ XUẤT FILE (DEPLOYMENT)
    # ---------------------------------------------------------
    print(f"\n[PHASE B] Huấn luyện Mô hình Thực chiến (Final Production Model)")
    final_train_start = end_date - pd.DateOffset(months=12)
    final_test_start = end_date - pd.DateOffset(months=2)
    
    i_tr_final = timestamps[(timestamps >= final_train_start) & (timestamps < final_test_start)].index.tolist()
    i_te_final = timestamps[timestamps >= final_test_start].index.tolist()
    
    print(f"🔥 Đang nung nóng mô hình với dữ liệu mới nhất ({final_train_start.date()} -> Hiện tại)...")
    auc_l, auc_n, auc_e, lgbm_final, nn_final, scalers = train_and_evaluate_window(df, i_tr_final, i_te_final, is_final_run=True)
    
    print(f"\n{'='*50}\n BẢNG XẾP HẠNG (HARD AUC - VÒNG CUỐI)\n{'='*50}")
    print(f" 1. LightGBM (Tabular Only)  : {auc_l:.4f}")
    print(f" 2. Transformer (Fine-tuned) : {auc_n:.4f}")
    print(f" 3. ENSEMBLE (Sự đồng thuận) : {auc_e:.4f}")
    print(f"{'='*50}")
    
    # Đóng gói và lưu
    import shutil
    tf_dir = Path(f"/kaggle/working/models/{tf}")
    tf_dir.mkdir(parents=True, exist_ok=True)
    
    joblib.dump(lgbm_final, tf_dir / 'ensemble_lgbm_tabular.joblib')
    torch.save(nn_final.state_dict(), tf_dir / 'ensemble_transformer.pth')
    
    meta_data = {
        'seq_features': SEQ_FEATURES, 'context_features': CONTEXT_FEATURES, 'signal_features': SIGNAL_FEATURES,
        'window_size': WINDOW_SIZE, 'ctx_scaler': scalers['ctx'], 'sig_scaler': scalers['sig'], 
        'seq_scaler': scalers['seq'], 'sym_encoder': scalers['sym'], 'num_symbols': len(scalers['sym'].classes_)
    }
    joblib.dump(meta_data, tf_dir / 'ensemble_meta.joblib')
    
    zip_filename = f"/kaggle/working/hybrid_production_{tf}"
    shutil.make_archive(zip_filename, 'zip', tf_dir)
    print(f"\n🚀 XONG! Sẵn sàng tải về: {zip_filename}.zip")
    
    from IPython.display import display, FileLink
    display(FileLink(f"hybrid_production_{tf}.zip"))

train_hybrid_lgbm(TIMEFRAME)
